# 🌳 Aula 13 — Árvores de Decisão e Comparação de Modelos

## Comparando algoritmos de classificação para tomar decisões

**Disciplina:** ISW-039 — Mineração de Dados  
**Curso:** Desenvolvimento de Software Multiplataforma (DSM)  
**Ambiente:** Google Colab  
**Linguagem:** Python  
**Bibliotecas:** Pandas, NumPy, Matplotlib e Scikit-learn

---

## 🎯 Objetivos da aula

Ao final desta aula, você deverá ser capaz de:

- Compreender o conceito de Árvore de Decisão;
- Entender como uma árvore transforma dados em regras;
- Treinar uma Árvore de Decisão com Python;
- Visualizar uma árvore;
- Interpretar as decisões do modelo;
- Identificar importância das variáveis;
- Comparar Árvore de Decisão e KNN;
- Avaliar diferentes modelos utilizando as mesmas métricas;
- Compreender que o melhor modelo depende do problema;
- Aplicar comparação de modelos ao projeto individual.

> **Projeto didático:** continuaremos utilizando o monitoramento de motores elétricos e o problema de classificação de **Normal × Falha**.


# 🧠 1. Relembrando a classificação

Na aula anterior construímos um classificador utilizando KNN.

Nosso fluxo era:

```text
Temperatura
Vibração
Corrente
Tensão
RPM
    ↓
  KNN
    ↓
Normal / Falha
```

Hoje vamos utilizar outro algoritmo:

> 🌳 **Árvore de Decisão**

A ideia será comparar os dois modelos utilizando os mesmos dados.


# 🌳 2. O que é uma Árvore de Decisão?

Uma Árvore de Decisão utiliza perguntas sucessivas para chegar a uma decisão.

Imagine um especialista analisando um motor:

```text
Temperatura > 75 °C?
        |
      SIM
        ↓
Vibração > 2.8?
     /       \
   SIM       NÃO
    ↓          ↓
  FALHA      NORMAL
```

O modelo aprende essas regras a partir dos dados de treinamento.

Uma vantagem importante é a **interpretabilidade**.

Em muitos casos conseguimos explicar:

> "O modelo classificou como falha porque a temperatura ultrapassou determinado valor e a vibração também estava elevada."


# 🔍 3. Estrutura de uma árvore

Uma árvore possui:

### Raiz
Primeira decisão.

### Nós
Perguntas ou divisões intermediárias.

### Ramos
Caminhos possíveis.

### Folhas
Resultado final.

Exemplo:

```text
                  Temperatura > 72?
                    /          \
                  SIM          NÃO
                  /              \
          Vibração > 2.5?       NORMAL
             /      \
           SIM      NÃO
           /          \
        FALHA        NORMAL
```

O algoritmo encontra automaticamente as divisões que melhor separam as classes.


# 💻 4. Preparando o ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

np.random.seed(42)

print("Ambiente preparado!")

# 📥 5. Criando a base de motores

Vamos utilizar uma base semelhante à aula anterior.



In [ ]:
n = 800

df = pd.DataFrame({
    "temperatura": np.random.normal(65, 8, n),
    "vibracao": np.random.normal(2.2, 0.7, n),
    "corrente": np.random.normal(13, 2, n),
    "tensao": np.random.normal(380, 4, n),
    "rpm": np.random.normal(1740, 15, n)
})

risco = (
    (df["temperatura"] > 76) &
    (df["vibracao"] > 2.8)
) | (
    (df["corrente"] > 16) &
    (df["temperatura"] > 70)
)

df["falha"] = risco.astype(int)

df.head()

# 🔎 6. Separando features e target

In [ ]:
X = df.drop("falha", axis=1)
y = df["falha"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

# ✂️ 7. Separando treinamento e teste

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treino:", X_treino.shape)
print("Teste:", X_teste.shape)

# 📏 8. Preparação para o KNN

O KNN depende de distância, por isso utilizaremos dados normalizados.

A Árvore de Decisão não depende da mesma forma da escala das variáveis.

Vamos manter:

```text
X_treino / X_teste
```

para a árvore e criar versões normalizadas para o KNN.


In [ ]:
scaler = StandardScaler()

X_treino_scaled = scaler.fit_transform(X_treino)
X_teste_scaled = scaler.transform(X_teste)

# 🤖 9. Modelo 1 — KNN

Vamos reproduzir o modelo da aula anterior.


In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_treino_scaled, y_treino)

pred_knn = knn.predict(X_teste_scaled)

print(classification_report(
    y_teste,
    pred_knn,
    target_names=["Normal", "Falha"],
    zero_division=0
))

# 🌳 10. Modelo 2 — Árvore de Decisão

Agora vamos criar nossa árvore.

Começaremos com:

```text
max_depth = 4
```

Esse parâmetro limita a profundidade da árvore.


In [ ]:
arvore = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

arvore.fit(X_treino, y_treino)

pred_arvore = arvore.predict(X_teste)

print("Árvore treinada!")

Vamos avaliar.


In [ ]:
print(classification_report(
    y_teste,
    pred_arvore,
    target_names=["Normal", "Falha"],
    zero_division=0
))

# 🌳 11. Visualizando a árvore

Uma das vantagens da Árvore de Decisão é a possibilidade de visualizar suas regras.


In [ ]:
plt.figure(figsize=(20, 10))

plot_tree(
    arvore,
    feature_names=X.columns,
    class_names=["Normal", "Falha"],
    filled=True,
    rounded=True,
    fontsize=9
)

plt.title("Árvore de Decisão — Classificação de Falhas")
plt.show()

Observe as perguntas realizadas pelo modelo.

Procure identificar:

- qual variável aparece na raiz;
- quais variáveis aparecem nos níveis seguintes;
- quais condições levam à classe Falha;
- quais condições levam à classe Normal.

Isso aproxima o modelo de uma **regra de decisão compreensível por humanos**.


# 🔎 12. Importância das variáveis

A árvore permite analisar a importância relativa das variáveis utilizadas nas divisões.


In [ ]:
importancias = pd.Series(
    arvore.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

importancias

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(importancias.index, importancias.values)
plt.title("Importância das Variáveis — Árvore de Decisão")
plt.xlabel("Variável")
plt.ylabel("Importância")
plt.xticks(rotation=30)
plt.show()

### Pergunta

Qual variável teve maior importância para a árvore?

Isso significa necessariamente que essa variável **causa** a falha?

> Não. Importância no modelo não significa causalidade.


# 🔲 13. Matriz de confusão da Árvore



In [ ]:
cm_arvore = confusion_matrix(y_teste, pred_arvore)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_arvore,
    display_labels=["Normal", "Falha"]
)

disp.plot()
plt.title("Matriz de Confusão — Árvore de Decisão")
plt.show()

Agora compare mentalmente com a matriz de confusão do KNN.

O objetivo não é apenas perguntar:

> "Qual tem maior acurácia?"

Precisamos perguntar:

> "Qual modelo comete os erros mais aceitáveis para o nosso problema?"


# 📊 14. Comparando os modelos

Vamos calcular várias métricas.


In [ ]:
metricas = pd.DataFrame({
    "Modelo": ["KNN", "Árvore de Decisão"],
    "Acurácia": [
        accuracy_score(y_teste, pred_knn),
        accuracy_score(y_teste, pred_arvore)
    ],
    "Precisão_Falha": [
        precision_score(y_teste, pred_knn, zero_division=0),
        precision_score(y_teste, pred_arvore, zero_division=0)
    ],
    "Recall_Falha": [
        recall_score(y_teste, pred_knn, zero_division=0),
        recall_score(y_teste, pred_arvore, zero_division=0)
    ],
    "F1_Falha": [
        f1_score(y_teste, pred_knn, zero_division=0),
        f1_score(y_teste, pred_arvore, zero_division=0)
    ]
})

metricas.round(3)

Agora temos uma comparação mais completa.

Não devemos escolher um modelo apenas porque possui uma métrica maior.

A escolha deve considerar o **problema de negócio**.


# 📈 15. Visualizando a comparação



In [ ]:
metricas_plot = metricas.set_index("Modelo")

metricas_plot.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.title("Comparação dos Modelos")
plt.xlabel("Modelo")
plt.ylabel("Valor da métrica")
plt.ylim(0, 1.05)
plt.xticks(rotation=0)
plt.legend(title="Métrica")
plt.show()

# ⚙️ 16. O problema do overfitting

Uma árvore muito profunda pode aprender detalhes específicos dos dados de treinamento.

Isso pode produzir:

```text
Treino → excelente
Teste  → ruim
```

Esse fenômeno é chamado de:

> **Overfitting (sobreajuste)**


Vamos observar o comportamento de árvores com diferentes profundidades.


In [ ]:
resultados_arvore = []

for profundidade in range(1, 11):
    modelo = DecisionTreeClassifier(
        max_depth=profundidade,
        random_state=42
    )

    modelo.fit(X_treino, y_treino)

    acc_treino = modelo.score(X_treino, y_treino)
    acc_teste = modelo.score(X_teste, y_teste)

    resultados_arvore.append({
        "profundidade": profundidade,
        "treino": acc_treino,
        "teste": acc_teste
    })

resultados_arvore_df = pd.DataFrame(resultados_arvore)

resultados_arvore_df

In [ ]:
plt.figure(figsize=(9, 5))

plt.plot(
    resultados_arvore_df["profundidade"],
    resultados_arvore_df["treino"],
    marker="o",
    label="Treinamento"
)

plt.plot(
    resultados_arvore_df["profundidade"],
    resultados_arvore_df["teste"],
    marker="o",
    label="Teste"
)

plt.title("Profundidade da Árvore × Desempenho")
plt.xlabel("Profundidade")
plt.ylabel("Acurácia")
plt.xticks(range(1, 11))
plt.legend()
plt.show()

Observe o comportamento das duas curvas.

Se a acurácia de treinamento continuar aumentando enquanto a de teste deixa de melhorar ou começa a cair, temos um sinal de sobreajuste.

Por isso, parâmetros como `max_depth` são importantes.


# 🔧 17. Comparando árvores com diferentes profundidades



In [ ]:
comparacao_profundidade = []

for profundidade in [2, 3, 4, 5, 6, 8, 10, None]:
    modelo = DecisionTreeClassifier(
        max_depth=profundidade,
        random_state=42
    )

    modelo.fit(X_treino, y_treino)
    pred = modelo.predict(X_teste)

    comparacao_profundidade.append({
        "max_depth": str(profundidade),
        "acuracia": accuracy_score(y_teste, pred),
        "precisao_falha": precision_score(y_teste, pred, zero_division=0),
        "recall_falha": recall_score(y_teste, pred, zero_division=0),
        "f1_falha": f1_score(y_teste, pred, zero_division=0)
    })

pd.DataFrame(comparacao_profundidade).round(3)

# 🧠 18. Interpretabilidade × desempenho

Compare:

### KNN

Vantagens:

- simples de entender;
- funciona bem em alguns problemas;
- fácil de implementar.

Desvantagens:

- depende de escala;
- pode ficar pesado com grandes volumes;
- sua explicação não é tão direta.

### Árvore de Decisão

Vantagens:

- regras interpretáveis;
- visualização;
- não exige normalização da mesma forma;
- identifica importância das variáveis.

Desvantagens:

- pode sofrer overfitting;
- pequenas mudanças nos dados podem alterar a árvore;
- uma árvore isolada pode não ser a melhor solução para todos os problemas.


# 🏭 19. Qual modelo escolher?

Imagine que temos:

```text
KNN
Acurácia: 95%
Recall Falha: 82%

Árvore
Acurácia: 94%
Recall Falha: 91%
```

Qual é melhor?

Se nosso objetivo principal for **não deixar falhas reais passarem despercebidas**, a Árvore pode ser mais interessante.

Portanto:

> **A melhor métrica depende do problema.**

E:

> **O melhor modelo depende do objetivo.**


# 🧪 20. Fazendo uma previsão com a árvore

Vamos utilizar o mesmo motor da aula anterior.


In [ ]:
novo_motor = pd.DataFrame({
    "temperatura": [82],
    "vibracao": [3.4],
    "corrente": [15.5],
    "tensao": [380],
    "rpm": [1735]
})

previsao_arvore = arvore.predict(novo_motor)

print("Classe prevista:", previsao_arvore[0])

if previsao_arvore[0] == 1:
    print("Resultado: possível FALHA")
else:
    print("Resultado: NORMAL")

Podemos também obter probabilidades estimadas pelo modelo.


In [ ]:
probabilidades = arvore.predict_proba(novo_motor)

print("Probabilidade de Normal:", round(probabilidades[0][0], 3))
print("Probabilidade de Falha:", round(probabilidades[0][1], 3))

Essas probabilidades não devem ser interpretadas automaticamente como "certeza de falha".

Elas são estimativas produzidas pelo modelo a partir das folhas alcançadas pela amostra.


# 📝 21. Exercícios

## Exercício 1 — Conceitos

Explique como uma Árvore de Decisão pode transformar dados em regras de decisão.


In [ ]:
# Sua resposta



## Exercício 2 — Treinamento

Crie uma Árvore de Decisão com:

```text
max_depth = 3
```

Calcule a acurácia.


In [ ]:
# Sua resposta



## Exercício 3 — Visualização

Visualize a árvore do exercício anterior.

Identifique:

- raiz;
- primeira variável utilizada;
- caminho até uma folha de Falha.


In [ ]:
# Sua resposta



## Exercício 4 — Importância

Mostre a importância das variáveis.

Qual foi a mais importante?


In [ ]:
# Sua resposta



## Exercício 5 — Profundidade

Teste:

```text
max_depth = 2
max_depth = 4
max_depth = 6
max_depth = 10
```

Compare o desempenho.


In [ ]:
# Sua resposta



## Exercício 6 — Overfitting

Observe as acurácias de treinamento e teste.

Em qual profundidade existe maior diferença entre elas?

O que isso pode indicar?


In [ ]:
# Sua resposta



## Exercício 7 — Comparação

Compare:

```text
KNN
Árvore de Decisão
```

Utilizando:

- acurácia;
- precisão;
- recall;
- F1-score.


In [ ]:
# Sua resposta



## Exercício 8 — Matriz de confusão

Gere a matriz de confusão da árvore.

Quantos falsos negativos foram encontrados?


In [ ]:
# Sua resposta



## Exercício 9 — Decisão

Se o custo de um falso negativo for muito alto, qual modelo você escolheria entre KNN e Árvore?

Utilize as métricas encontradas para justificar.


In [ ]:
# Sua resposta



## Exercício 10 — Nova previsão

Crie três novos registros de motores e utilize:

1. KNN;
2. Árvore de Decisão.

Compare as previsões.


In [ ]:
# Sua resposta



# 🚀 22. Desafio — Comparação de modelos no seu projeto

Agora vamos levar a comparação para o **projeto individual**.

Se o seu problema for de classificação:

### 1. Prepare os dados

Defina:

```text
Features:
________________________

Target:
________________________
```

### 2. Separe treino e teste

Utilize uma divisão adequada.

### 3. Crie dois modelos

Modelo 1:

```text
KNN
```

Modelo 2:

```text
Árvore de Decisão
```

### 4. Avalie

Para cada modelo apresente:

- acurácia;
- precisão;
- recall;
- F1-score;
- matriz de confusão.

### 5. Compare

Crie uma tabela:

| Modelo | Acurácia | Precisão | Recall | F1 |
|---|---:|---:|---:|---:|
| KNN | | | | |
| Árvore | | | | |

### 6. Conclusão

Responda:

> **Qual modelo é mais adequado para o meu problema e por quê?**

Não basta dizer que possui maior acurácia.

Você deverá justificar considerando o objetivo do projeto.


In [ ]:
# Desenvolva a comparação dos modelos do seu projeto aqui.



# 🏭 23. Aplicação no projeto didático

No exemplo industrial, já temos uma evolução significativa:

```text
AULA 11
Clustering
↓
Descobrimos grupos sem rótulos

AULA 12
KNN
↓
Aprendemos a prever Normal/Falha

AULA 13
Árvore de Decisão
↓
Criamos regras e comparamos modelos
```

Agora nosso projeto possui uma estrutura mais próxima de uma solução real:

```text
DADOS
  ↓
PRÉ-PROCESSAMENTO
  ↓
ANÁLISE EXPLORATÓRIA
  ↓
VISUALIZAÇÃO
  ↓
MODELAGEM
  ↓
COMPARAÇÃO
  ↓
AVALIAÇÃO
  ↓
DECISÃO
```


# 📌 24. Checklist da Aula

- [ ] Entendo Árvore de Decisão;
- [ ] Sei explicar raiz, nós, ramos e folhas;
- [ ] Sei treinar uma árvore;
- [ ] Sei controlar `max_depth`;
- [ ] Sei visualizar uma árvore;
- [ ] Sei analisar importância das variáveis;
- [ ] Entendo overfitting;
- [ ] Sei comparar treinamento e teste;
- [ ] Sei comparar KNN e Árvore;
- [ ] Sei utilizar matriz de confusão;
- [ ] Sei analisar precisão, recall e F1;
- [ ] Entendo que o melhor modelo depende do problema;
- [ ] Consigo comparar modelos no meu projeto.

---

# 🎯 Conclusão

A sequência da disciplina está agora:

```text
Aula 03 → Pandas
Aula 04 → Limpeza
Aula 05 → ETL
Aula 06 → Web Scraping
Aula 07 → Banco de Dados + SQL
Aula 08 → Análise Exploratória
Aula 09 → Amostragem + Balanceamento
Aula 10 → Visualização
Aula 11 → Clustering
Aula 12 → Classificação
Aula 13 → Árvores de Decisão + Comparação
```

Até aqui os alunos já passaram por um ciclo completo:

```text
DADOS
 ↓
PREPARAÇÃO
 ↓
EXPLORAÇÃO
 ↓
VISUALIZAÇÃO
 ↓
AGRUPAMENTO
 ↓
CLASSIFICAÇÃO
 ↓
COMPARAÇÃO DE MODELOS
```

Na próxima aula avançaremos para **Regressão e previsão de valores numéricos**, ampliando a aplicação da aprendizagem supervisionada.

> 📈 **Próxima aula: Regressão — prevendo valores contínuos.**
